# Inference: Meditation vs Thinking Classifier

This notebook demonstrates how to:
1. Load a trained classifier model
2. Extract HRV features from new subject data
3. Make predictions with class labels and probabilities

**Model trained on:**
- Think (Label 0) vs Meditate (Label 1)
- 8 HRV features from ECG signal

## 1. Setup and Imports

In [ ]:
import numpy as np
import os
from pathlib import Path
from dotenv import load_dotenv
import pandas as pd

# Load environment variables
env_path = Path('.').resolve() / '.env'
load_dotenv(env_path)

# Import EEG processing modules
from eeg_processor import Config, DataLoader, HRVFeatureExtractor, HRVClassifier

print(f"Environment loaded from: {env_path}")
print(f"BIDS Root: {os.getenv('BIDS_ROOT')}")

## 2. Load Trained Model

In [ ]:
# Initialize configuration
config = Config(bids_root=os.getenv('BIDS_ROOT'), random_seed=42)

# Load trained model
model_dir = Path('.').resolve() / 'models' / 'meditation_classifier'

if not model_dir.exists():
    print(f"[ERROR] Model not found at {model_dir}")
    print("Please train the model first using train_classifier.ipynb")
else:
    classifier = HRVClassifier(input_size=8)
    classifier.load(str(model_dir))
    print(f"[OK] Model loaded from {model_dir}")
    print(f"Device: {classifier.device}")

## 3. Explore Available Subjects

In [ ]:
# Explore BIDS structure to see available subjects
loader = DataLoader(config)
bids_info = loader.explore_bids_structure()

print(f"\nAvailable subjects: {loader.subjects}")

## 4. Feature Extraction Helper

In [ ]:
def extract_hrv_features_for_subject(subject_id: str, loader: DataLoader, config: Config) -> dict:
    """
    Extract HRV features from a subject's EEG data.
    
    Parameters
    ----------
    subject_id : str
        Subject ID to load
    loader : DataLoader
        DataLoader instance
    config : Config
        Configuration object
    
    Returns
    -------
    dict or None
        Dictionary with HRV features, or None if extraction failed
    """
    try:
        # Load data
        raw = loader.load_eeg_data(subject_id, session=None, task=None)
        
        if raw is None:
            print(f"[ERROR] Could not load data for subject {subject_id}")
            return None
        
        # Set up channels
        loader.setup_montage(montage_name='biosemi64', on_missing='ignore')
        channel_type_mapping = {
            'EXG1': 'eog', 'EXG2': 'eog', 'EXG3': 'eog', 'EXG4': 'eog',
            'EXG5': 'misc', 'EXG6': 'misc', 'EXG7': 'ecg'
        }
        loader.set_channel_types(channel_type_mapping)
        
        # Extract HRV features
        ecg_idx = raw.ch_names.index('EXG7')
        ecg_signal = raw.get_data(picks=ecg_idx)[0]
        
        extractor = HRVFeatureExtractor(sampling_rate=int(raw.info['sfreq']))
        features, _ = extractor.extract_with_interval(ecg_signal)
        
        return features, raw.info['sfreq']
    
    except Exception as e:
        print(f"[ERROR] Exception for subject {subject_id}: {e}")
        return None, None

print("Helper function defined.")

## 5. Single Subject Prediction

In [ ]:
# Select a subject to predict
subject_id = loader.subjects[0]  # First available subject

print(f"\nPredicting for subject: {subject_id}")
print("="*60)

# Extract features
result = extract_hrv_features_for_subject(subject_id, loader, config)

if result[0] is not None:
    features, sfreq = result
    
    # Convert to array format for prediction
    X = np.array([list(features.values())])
    
    # Make prediction with probabilities
    pred_class, pred_probs = classifier.predict_with_proba(X)
    
    class_name = "Think" if pred_class[0] == 0 else "Meditate"
    think_prob = pred_probs[0][0]
    meditate_prob = pred_probs[0][1]
    confidence = max(pred_probs[0])
    
    print(f"\nSubject: {subject_id}")
    print(f"Sampling Rate: {int(sfreq)} Hz")
    print(f"\nPredicted class:    {class_name} ({pred_class[0]})")
    print(f"\nProbabilities:")
    print(f"  - Think (0):    {think_prob:.4f}")
    print(f"  - Meditate (1): {meditate_prob:.4f}")
    print(f"\nConfidence: {confidence:.4f}")
    
    # Show extracted features
    print(f"\nExtracted HRV Features:")
    for fname, fvalue in features.items():
        print(f"  {fname}: {fvalue:.4f}")

## 6. Batch Prediction for Multiple Subjects

In [ ]:
# Predict for multiple subjects
subjects_to_predict = loader.subjects[:5]  # First 5 subjects

print(f"\nMaking predictions for {len(subjects_to_predict)} subjects...\n")

results = []

for subject_id in subjects_to_predict:
    result = extract_hrv_features_for_subject(subject_id, loader, config)
    
    if result[0] is not None:
        features, sfreq = result
        
        # Make prediction
        X = np.array([list(features.values())])
        pred_class, pred_probs = classifier.predict_with_proba(X)
        
        class_name = "Think" if pred_class[0] == 0 else "Meditate"
        
        results.append({
            'Subject': subject_id,
            'Predicted Class': class_name,
            'Class Index': pred_class[0],
            'Think Probability': pred_probs[0][0],
            'Meditate Probability': pred_probs[0][1],
            'Confidence': max(pred_probs[0])
        })

# Create DataFrame for better visualization
results_df = pd.DataFrame(results)
print(results_df.to_string(index=False))

## 7. Visualization of Results

In [ ]:
import matplotlib.pyplot as plt

if len(results) > 0:
    fig, axes = plt.subplots(1, 2, figsize=(14, 5))
    
    # Probability bars
    subjects = results_df['Subject'].values
    think_probs = results_df['Think Probability'].values
    meditate_probs = results_df['Meditate Probability'].values
    
    x = np.arange(len(subjects))
    width = 0.35
    
    axes[0].bar(x - width/2, think_probs, width, label='Think', alpha=0.8)
    axes[0].bar(x + width/2, meditate_probs, width, label='Meditate', alpha=0.8)
    axes[0].set_ylabel('Probability', fontsize=12)
    axes[0].set_title('Prediction Probabilities', fontsize=14, fontweight='bold')
    axes[0].set_xticks(x)
    axes[0].set_xticklabels(subjects)
    axes[0].legend(fontsize=11)
    axes[0].grid(axis='y', alpha=0.3)
    
    # Confidence scores
    colors = ['#1f77b4' if c == 0 else '#ff7f0e' for c in results_df['Class Index'].values]
    axes[1].bar(subjects, results_df['Confidence'].values, color=colors, alpha=0.8)
    axes[1].set_ylabel('Confidence', fontsize=12)
    axes[1].set_title('Prediction Confidence', fontsize=14, fontweight='bold')
    axes[1].set_ylim([0, 1])
    axes[1].grid(axis='y', alpha=0.3)
    
    # Add legend for colors
    from matplotlib.patches import Patch
    legend_elements = [Patch(facecolor='#1f77b4', alpha=0.8, label='Think'),
                      Patch(facecolor='#ff7f0e', alpha=0.8, label='Meditate')]
    axes[1].legend(handles=legend_elements, fontsize=11)
    
    plt.tight_layout()
    plt.show()

## 8. Custom Prediction Example

You can also use the classifier directly with pre-extracted features:

In [ ]:
# Example: Make a prediction with manually provided features
# These would be 8 HRV features: rr_mean, sdnn, rmssd, pnn50, lf, hf, lf_hf, vlf

example_features = np.array([
    [800.0, 50.0, 30.0, 20.0, 100.0, 50.0, 2.0, 10.0]  # Example HRV values
])

pred_class, pred_probs = classifier.predict_with_proba(example_features)

print("Example Prediction (with custom features):")
print(f"  Predicted class:      {'Think' if pred_class[0] == 0 else 'Meditate'} ({pred_class[0]})")
print(f"  Think probability:    {pred_probs[0][0]:.4f}")
print(f"  Meditate probability: {pred_probs[0][1]:.4f}")
print(f"  Confidence:           {max(pred_probs[0]):.4f}")

## 9. Summary

The inference pipeline:
1. Loads the trained classifier model
2. Extracts HRV features from ECG signals
3. Makes predictions with probabilities
4. Returns both class label and confidence scores

**Output format:**
- Predicted class: 0 (Think) or 1 (Meditate)
- Probability[0]: Probability of Think class
- Probability[1]: Probability of Meditate class
- Confidence: Maximum probability score

In [ ]:
print("\n" + "="*60)
print("INFERENCE COMPLETE")
print("="*60)
print(f"\nModel: {model_dir}")
print(f"Total predictions made: {len(results)}")
print(f"\nModel ready for production inference!")